In [97]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [98]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [99]:
# --- INITIALIZATION ---
num_bits = 24
num_test_bits = 8
error_threshold = 0.20 # 20% threshold for attack detection
test_indices = range(num_test_bits)
key_indices = range(num_test_bits, num_bits)

In [100]:
# --- HELPER: QUANTUM RANDOMNESS ---
def get_quantum_random_bits(n):
  """Generates n random bits using quantum measurement."""
  qc = QuantumCircuit(n)
  qc.h(range(n))
  qc.measure_all()
  backend = BasicSimulator()
  job = backend.run(transpile(qc, backend), shots=1)
  bitstring = list(job.result().get_counts().keys())[0]
  return [int(b) for b in bitstring[::-1]]

# --- HELPER: FORMATTING ---
def format_as_table(headers, data_lists):
  """
  Formats lists into a horizontal table where headers are the first
  element of each row.
  """
  if not headers or not data_lists:
    return ""

  # 1. Determine the width needed for the header column
  header_width = max(len(h) for h in headers)

  # 2. Determine the width for each data cell to ensure alignment
  # We flatten the lists to find the longest string representation of any data point
  all_items = [str(item) for sublist in data_lists for item in sublist]
  cell_width = max(len(item) for item in all_items) if all_items else 1

  table_lines = []

  # 3. Build each row
  for header, row_data in zip(headers, data_lists):
    # Start the row with the header label
      row_str = f"{header:<{header_width}} |"
      # Add each data point from the list to the row
      for item in row_data:
        if header in ["Alice's bases",  "Eve's bases",  "Bob's bases",]:
          item = "s" if item == 0 else "d"
        row_str += f" {str(item):<{cell_width}} |"
      table_lines.append(row_str)
      if header == "Index":
        table_lines.append("-" * len(row_str))

  return "\n".join(table_lines)

# Helper function to format qubit state
def get_qubit_state(bits, bases):
  qubit_state = []
  if len(bits) != len(bases):
    return "Invalid input"
  for i in range(len(bits)):
    if bases[i]==0:
      qubit_state.append(f"0") if bits[i]==0 else qubit_state.append(f"1")
    else:
      qubit_state.append(f"+") if bits[i]==0 else qubit_state.append(f"-")

  return qubit_state

In [101]:
# --- 1. ALICE'S BLOCK ---
alice_bits = get_quantum_random_bits(num_bits)
alice_bases = get_quantum_random_bits(num_bits) # 0: Z, 1: X

# Alice prepares her qubits
quantum_channel = []
for bit, basis in zip(alice_bits, alice_bases):
  qc = QuantumCircuit(1, 1)
  if bit == 1: qc.x(0)
  if basis == 1: qc.h(0)
  quantum_channel.append(qc)

qubit_state = get_qubit_state(alice_bits, alice_bases)

In [102]:
# --- 2. EVE'S ATTACK ---
# Eve intercepts the qubits, measures them, and sends them on
eve_bases = get_quantum_random_bits(num_bits)
eve_results = []

for i in range(num_bits):
  qc = quantum_channel[i]
  if eve_bases[i] == 1: qc.h(0) # Eve chooses diagonal
  qc.measure(0, 0)

  # Simulate Eve's measurement
  backend = BasicSimulator()
  res = backend.run(transpile(qc, backend), shots=1).result().get_counts()
  eve_bit = int(list(res.keys())[0])
  eve_results.append(eve_bit)

  # Eve must "reset" the qubit state to what she measured to send to Bob
  new_qc = QuantumCircuit(1, 1)
  if eve_bit == 1: new_qc.x(0)
  if eve_bases[i] == 1: new_qc.h(0)
  quantum_channel[i] = new_qc

In [103]:
# --- 3. BOB'S BLOCK ---
bob_bases = get_quantum_random_bits(num_bits)
bob_results = []

for i in range(num_bits):
  qc = quantum_channel[i]
  if bob_bases[i] == 1: qc.h(0)
  qc.measure(0, 0)
  backend = BasicSimulator()
  res = backend.run(transpile(qc, backend), shots=1).result().get_counts()
  bob_results.append(int(list(res.keys())[0]))

In [104]:
# --- 4. SIFTING AND DETECTION ---
# 1. Sifting for test bits (to detect Eve)
sifted_test_indices = [i for i in test_indices if alice_bases[i] == bob_bases[i]]
alice_test_sample = [alice_bits[i] for i in sifted_test_indices]
bob_test_sample = [bob_results[i] for i in sifted_test_indices]

# 2. Sifting for key bits
sifted_key_indices = [i for i in key_indices if alice_bases[i] == bob_bases[i]]
shared_key = [bob_results[i] for i in sifted_key_indices]

# Calculate error rate on test sample only
errors = sum(1 for a, b in zip(alice_test_sample, bob_test_sample) if a != b)
error_rate = errors / len(alice_test_sample) if alice_test_sample else 0

purpose = ["Test" if i < num_test_bits else "Key" for i in range(num_bits)]

ismatching = []

for i in range(num_bits):
  if alice_bases[i] == bob_bases[i]:
    ismatching.append("Y")
  else:
    ismatching.append("")

In [105]:
# --- OUTPUT ---
headers=["Index",
  "Purpose",
  "Alice's initial bits",
  "Alice's bases",
  "Qubit state",
  "Eve's bases",
  "Eve's results",
  "Bob's bases",
  "Bob's results",
  "Matching bases (Y if match)"]
data = [indices, purpose, alice_bits, alice_bases, qubit_state, eve_bases, eve_results, bob_bases, bob_results, ismatching]
print(format_as_table(headers, data))

print("-" * 30)
print(f"Test Sample Size (matched bases): {len(alice_test_sample)}")
print(f"Errors in Test Sample:           {errors}")
print(f"Error Rate:    {error_rate:.2%}")
print(f"Potential Shared Key Length:      {len(shared_key)}")

if error_rate > error_threshold:
  print("RESULT: ATTACK DETECTED! Key discarded.")
else:
  print("RESULT: Secure transmission. Key accepted.")
  print(f"Final Key: {shared_key}")

# Reflection:
# Based on the results of this assignment, I observed that 24 bits length is insufficient to:
# 1) detect the attack by Eve and 2) produce a appropriate-sized binary one-time pad
# Modern day systems usually produce 256-bit keys
# However the maximum bit length allowed for BasicSimulator was 24 so I stuck to using it since it was provided in the imports list

Index                       | 0    | 1    | 2    | 3    | 4    | 5    | 6    | 7    | 8    | 9    | 10   | 11   | 12   | 13   | 14   | 15   | 16   | 17   | 18   | 19   | 20   | 21   | 22   | 23   |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Purpose                     | Test | Test | Test | Test | Test | Test | Test | Test | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  | Key  |
Alice's initial bits        | 0    | 1    | 1    | 0    | 1    | 0    | 1    | 1    | 0    | 0    | 0    | 1    | 1    | 1    | 0    | 0    | 1    | 1    | 0    | 1    | 0    | 0    | 0    | 0    |
Alice's bases               | s    | d    | s    | s    | d    | d    | s    | d    | s    | s    | s    | d    | d    | d    | s    | s    | d    | d    | s    | s    | d    | d    | d    | d    |
Qubit stat